In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# 0. Device and seed
# ============================================================

torch.set_default_dtype(torch.float64)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook. Please run it in a CUDA-enabled PyTorch environment.")

device = torch.device("cuda")
print("Using device:", device)

torch.manual_seed(42)


# ============================================================
# 1. Problem setting
# ============================================================

n = 4
m = 2
output_dim = n + m

mu = 1.0
rho = 0.5
T = 10.0

epochs = 200
batch_size = 10

learning_rate = 1e-2
optimizer_kwargs = {"weight_decay": 0.0}

n_qubits = output_dim
L = 2
entangling_layers_per_block = 2

fc_hidden_dim = 16
fc_depth = 1


def generate_full_row_rank_A(m, n):
    assert m <= n, "Require m <= n."

    while True:
        A_cpu = torch.randn(m, n, dtype=torch.float64)
        if torch.linalg.matrix_rank(A_cpu).item() == m:
            return A_cpu.to(device)


A = generate_full_row_rank_A(m, n)
A = A / torch.linalg.norm(A, dim=1, keepdim=True)

c = torch.randn(n, 1, device=device)

x_feasible = torch.randn(n, 1, device=device)
b = A @ x_feasible

x0 = torch.zeros(n, 1, device=device)
v0 = torch.zeros(m, 1, device=device)
y0 = torch.cat([x0, v0], dim=0).T

print("A shape:", A.shape)
print("b shape:", b.shape)
print("c shape:", c.shape)
print("y0 shape:", y0.shape)


# ============================================================
# 2. Objective and gradient
# ============================================================

def objective_f(x):
    return 0.5 * mu * torch.sum((x - c.T) ** 2, dim=1)


def grad_f_manual(x):
    return mu * (x - c.T)


# ============================================================
# 3. Closed-form KKT reference
# ============================================================

with torch.no_grad():
    AAT = A @ A.T
    rhs = mu * (A @ c - b)

    v_star = torch.linalg.solve(AAT, rhs)
    x_star = c - (1.0 / mu) * A.T @ v_star

    ref_obj = objective_f(x_star.T).item()

print("\nReference objective f(x_star):", ref_obj)


# ============================================================
# 4. Pure torch statevector utilities
# ============================================================

state_dim = 2 ** n_qubits


def initial_state(batch_size):
    state = torch.zeros(batch_size, state_dim, dtype=torch.complex128, device=device)
    state[:, 0] = 1.0 + 0.0j
    return state


def rz_gate(theta):
    theta_c = theta.to(torch.complex128)
    gate = torch.zeros(theta.shape + (2, 2), dtype=torch.complex128, device=theta.device)
    gate[..., 0, 0] = torch.exp(-0.5j * theta_c)
    gate[..., 1, 1] = torch.exp(0.5j * theta_c)
    return gate


def ry_gate(theta):
    cos = torch.cos(0.5 * theta)
    sin = torch.sin(0.5 * theta)
    gate = torch.zeros(theta.shape + (2, 2), dtype=torch.complex128, device=theta.device)
    gate[..., 0, 0] = cos.to(torch.complex128)
    gate[..., 0, 1] = -sin.to(torch.complex128)
    gate[..., 1, 0] = sin.to(torch.complex128)
    gate[..., 1, 1] = cos.to(torch.complex128)
    return gate


def apply_single_qubit_gate(state, gate, wire):
    batch_size = state.shape[0]
    state_nd = state.reshape(batch_size, *([2] * n_qubits))
    perm = [0, wire + 1] + [idx + 1 for idx in range(n_qubits) if idx != wire]
    inv_perm = [0] * len(perm)
    for idx, value in enumerate(perm):
        inv_perm[value] = idx

    front = state_nd.permute(perm).reshape(batch_size, 2, -1)
    if gate.dim() == 2:
        front = torch.einsum("ij,bjk->bik", gate, front)
    else:
        front = torch.einsum("bij,bjk->bik", gate, front)

    state_nd = front.reshape(batch_size, 2, *([2] * (n_qubits - 1))).permute(inv_perm)
    return state_nd.reshape(batch_size, state_dim)


def cnot_indices(control, target):
    idx = torch.arange(state_dim, device=device)
    control_bit = n_qubits - 1 - control
    target_bit = n_qubits - 1 - target
    flip = ((idx >> control_bit) & 1).bool()
    perm = idx.clone()
    perm[flip] = perm[flip] ^ (1 << target_bit)
    return perm


cnot_perms = [cnot_indices(j, (j + 1) % n_qubits) for j in range(n_qubits)]


def apply_cnot_ring(state):
    for perm in cnot_perms:
        state = state[:, perm]
    return state


def apply_rot_layer(state, layer_weights):
    for sublayer in range(layer_weights.shape[0]):
        for wire in range(n_qubits):
            phi, theta, omega = layer_weights[sublayer, wire]
            state = apply_single_qubit_gate(state, rz_gate(phi), wire)
            state = apply_single_qubit_gate(state, ry_gate(theta), wire)
            state = apply_single_qubit_gate(state, rz_gate(omega), wire)
        state = apply_cnot_ring(state)
    return state


def apply_exp_encoding(state, t, layer):
    tau = t.reshape(-1) / T
    coef = 3 ** layer
    for wire in range(n_qubits):
        phi = coef * (wire + 1) * torch.pi * tau
        state = apply_single_qubit_gate(state, rz_gate(phi), wire)
        state = apply_single_qubit_gate(state, ry_gate(phi), wire)
    return state


def local_p0_features(state):
    probabilities = state.abs().square().reshape(state.shape[0], *([2] * n_qubits))
    features = []
    for wire in range(n_qubits):
        p0 = probabilities.select(dim=wire + 1, index=0).sum(dim=tuple(range(1, n_qubits)))
        features.append(2.0 * p0 - 1.0)
    return torch.stack(features, dim=1)


# ============================================================
# 5. QFM-QPINN and fully-connected PINN models
# ============================================================

class QFM_QPINN(nn.Module):
    def __init__(self):
        super().__init__()

        self.weights = nn.Parameter(
            0.2 * torch.randn(
                L + 1,
                entangling_layers_per_block,
                n_qubits,
                3,
                dtype=torch.float64,
                device=device,
            )
        )

        self.readout = nn.Linear(
            n_qubits,
            output_dim,
            bias=True,
            dtype=torch.float64,
        )

        nn.init.xavier_uniform_(self.readout.weight)
        nn.init.zeros_(self.readout.bias)

        self.alpha = nn.Parameter(
            torch.tensor(0.5, dtype=torch.float64, device=device)
        )

    def torch_qfm_features(self, t):
        state = initial_state(t.shape[0])
        state = apply_rot_layer(state, self.weights[0])

        for layer in range(L):
            state = apply_exp_encoding(state, t, layer)
            state = apply_rot_layer(state, self.weights[layer + 1])

        return local_p0_features(state)

    def forward(self, t):
        q_out = self.torch_qfm_features(t)
        raw = self.readout(q_out)

        alpha_pos = torch.nn.functional.softplus(self.alpha)
        envelope = 1.0 - torch.exp(-alpha_pos * t)

        return y0 + envelope * raw


class FC_PINN(nn.Module):
    def __init__(self, hidden_dim=16, depth=1):
        super().__init__()

        layers = []
        in_dim = 1

        for _ in range(depth):
            layers.append(nn.Linear(in_dim, hidden_dim, dtype=torch.float64))
            layers.append(nn.Tanh())
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, output_dim, dtype=torch.float64))

        self.net = nn.Sequential(*layers)

        self.alpha = nn.Parameter(
            torch.tensor(0.5, dtype=torch.float64, device=device)
        )

    def forward(self, t):
        tau = t / T
        raw = self.net(tau)

        alpha_pos = torch.nn.functional.softplus(self.alpha)
        envelope = 1.0 - torch.exp(-alpha_pos * t)

        y = y0 + envelope * raw

        return y


# ============================================================
# 7. Time derivative and QPINN loss
# ============================================================

def time_derivative(y, t):
    dy_dt = torch.zeros_like(y)

    for i in range(y.shape[1]):
        dy_dt[:, i:i + 1] = torch.autograd.grad(
            y[:, i:i + 1],
            t,
            grad_outputs=torch.ones_like(y[:, i:i + 1]),
            create_graph=True,
            retain_graph=True,
        )[0]

    return dy_dt


def qpinn_loss(model, t):
    y = model(t)

    x = y[:, :n]
    v = y[:, n:]

    dy_dt = time_derivative(y, t)

    dx_dt = dy_dt[:, :n]
    dv_dt = dy_dt[:, n:]

    grad_f = grad_f_manual(x)

    ATv = v @ A
    Ax_minus_b = x @ A.T - b.T

    r_x = dx_dt + rho * (grad_f + ATv)
    r_v = dv_dt - rho * Ax_minus_b

    loss_x = torch.mean(r_x ** 2)
    loss_v = torch.mean(r_v ** 2)

    loss = loss_x + loss_v

    return loss, loss_x, loss_v


# ============================================================
# 8. Evaluation helpers
# ============================================================

def evaluate_terminal_solution(model):
    with torch.no_grad():
        t_T = torch.tensor([[T]], dtype=torch.float64, device=device)
        y_T = model(t_T)

        x_T = y_T[:, :n].T
        v_T = y_T[:, n:].T

        x_error = torch.norm(x_T - x_star).item()
        objective = objective_f(x_T.T).item()
        objective_gap = objective - ref_obj
        feasibility_norm = torch.norm(A @ x_T - b).item()

    return {
        "x_T": x_T,
        "v_T": v_T,
        "x_error": x_error,
        "objective": objective,
        "objective_gap": objective_gap,
        "feasibility_norm": feasibility_norm,
    }


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def safe_for_log(v, eps=1e-16):
    v = float(v)
    if v <= 0.0:
        return eps
    return v


def make_time_batches(num_epochs, batch_size, seed):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    batches = []
    for _ in range(num_epochs):
        t = torch.rand(
            batch_size,
            1,
            dtype=torch.float64,
            generator=generator,
        ).to(device) * T
        batches.append(t)

    return batches


train_batches = make_time_batches(epochs, batch_size, seed=2026)
validation_t = torch.linspace(
    0.0,
    T,
    25,
    dtype=torch.float64,
    device=device,
).reshape(-1, 1)


def residual_on_grid(model, t_grid):
    t = t_grid.detach().clone().requires_grad_(True)
    loss, loss_x, loss_v = qpinn_loss(model, t)
    return loss.item(), loss_x.item(), loss_v.item()


# ============================================================
# 9. Lamb optimizer and training
# ============================================================

class Lamb(optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-6, weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            lr = group["lr"]
            eps = group["eps"]
            weight_decay = group["weight_decay"]

            for param in group["params"]:
                if param.grad is None:
                    continue

                grad = param.grad
                state = self.state[param]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(param)
                    state["exp_avg_sq"] = torch.zeros_like(param)

                exp_avg = state["exp_avg"]
                exp_avg_sq = state["exp_avg_sq"]
                state["step"] += 1

                exp_avg.mul_(beta1).add_(grad, alpha=1.0 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)

                bias_correction1 = 1.0 - beta1 ** state["step"]
                bias_correction2 = 1.0 - beta2 ** state["step"]
                adam_step = (exp_avg / bias_correction1) / (
                    (exp_avg_sq / bias_correction2).sqrt().add(eps)
                )
                if weight_decay != 0.0:
                    adam_step = adam_step.add(param, alpha=weight_decay)

                weight_norm = torch.linalg.vector_norm(param)
                adam_norm = torch.linalg.vector_norm(adam_step)
                if weight_norm > 0 and adam_norm > 0:
                    trust_ratio = weight_norm / adam_norm
                else:
                    trust_ratio = 1.0

                param.add_(adam_step * trust_ratio, alpha=-lr)

        return loss


optimizer_cls = getattr(optim, "Lamb", Lamb)


def make_optimizer(model):
    return optimizer_cls(model.parameters(), lr=learning_rate, **optimizer_kwargs)


def train_model(model, name, batches):
    optimizer = make_optimizer(model)

    history = {
        "loss": [],
        "loss_x": [],
        "loss_v": [],
        "x_error": [],
        "objective_gap": [],
        "feasibility_norm": [],
    }

    print(f"\nStart training {name} with {optimizer_cls.__name__}, lr={learning_rate:.1e}...")

    for epoch, t_base in enumerate(batches, start=1):
        optimizer.zero_grad()

        t = t_base.detach().clone().requires_grad_(True)
        loss, loss_x, loss_v = qpinn_loss(model, t)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        info = evaluate_terminal_solution(model)

        history["loss"].append(safe_for_log(loss.item()))
        history["loss_x"].append(safe_for_log(loss_x.item()))
        history["loss_v"].append(safe_for_log(loss_v.item()))
        history["x_error"].append(safe_for_log(info["x_error"]))
        history["objective_gap"].append(abs(info["objective_gap"]) + 1e-16)
        history["feasibility_norm"].append(safe_for_log(info["feasibility_norm"]))

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"{name} | Epoch {epoch:03d}/{len(batches)}, "
                f"loss={loss.item():.6e}, "
                f"loss_x={loss_x.item():.6e}, "
                f"loss_v={loss_v.item():.6e}, "
                f"||x(T)-x*||={info['x_error']:.6e}"
            )

    print(f"Finished training {name}.")

    return history


# Use separate seeds for the two architectures, then train both with the same
# optimizer and learning rate.
torch.manual_seed(123)
qfm_base = QFM_QPINN().to(device)

torch.manual_seed(456)
fc_base = FC_PINN(hidden_dim=fc_hidden_dim, depth=fc_depth).to(device)

qfm_params = count_parameters(qfm_base)
fc_params = count_parameters(fc_base)

print("\n================ Model Sizes ================")
print(f"QFM-QPINN trainable parameters: {qfm_params}")
print(f"FC-PINN trainable parameters:  {fc_params}")
print(f"Parameter ratio FC / QFM:      {fc_params / qfm_params:.2f}")

qfm_model = qfm_base
fc_model = fc_base

histories = {
    "QFM-QPINN": train_model(qfm_model, "QFM-QPINN", train_batches),
    "FC-PINN": train_model(fc_model, "FC-PINN", train_batches),
}

models = {
    "QFM-QPINN": qfm_model,
    "FC-PINN": fc_model,
}

final_infos = {name: evaluate_terminal_solution(model) for name, model in models.items()}

print("\n================ Final Comparison ================")
print("\nReference x_star:")
print(x_star.T.cpu())
print("Reference objective:", ref_obj)

for name, info in final_infos.items():
    print(f"\n{name} approximate x(T):")
    print(info["x_T"].T.cpu())
    print(f"{name} objective gap       = {info['objective_gap']:.12e}")
    print(f"{name} feasibility norm    = {info['feasibility_norm']:.12e}")
    print(f"{name} ||x(T) - x_star||   = {info['x_error']:.12e}")


# ============================================================
# 10. Publication-style plots
# ============================================================

def moving_average(values, window=9):
    values = torch.tensor(values, dtype=torch.float64)
    if values.numel() < window:
        return values.numpy()

    pad_left = window // 2
    pad_right = window - 1 - pad_left
    padded = torch.nn.functional.pad(values, (pad_left, pad_right), mode="replicate")
    kernel = torch.ones(window, dtype=torch.float64) / window
    smooth = torch.nn.functional.conv1d(
        padded.reshape(1, 1, -1),
        kernel.reshape(1, 1, -1),
    ).reshape(-1)

    return smooth.numpy()


plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
    "font.size": 11,
})

colors = {
    "QFM-QPINN": "#3561A7",
    "FC-PINN": "#C44E52",
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
epoch_axis = range(1, epochs + 1)

for name, hist in histories.items():
    color = colors[name]

    axes[0].semilogy(
        epoch_axis,
        hist["loss"],
        color=color,
        alpha=0.18,
        linewidth=1.0,
    )
    axes[0].semilogy(
        epoch_axis,
        moving_average(hist["loss"]),
        color=color,
        linewidth=2.4,
        label=f"{name} (lr={learning_rate:.0e})",
    )

    axes[1].semilogy(
        epoch_axis,
        hist["x_error"],
        color=color,
        alpha=0.18,
        linewidth=1.0,
    )
    axes[1].semilogy(
        epoch_axis,
        moving_average(hist["x_error"]),
        color=color,
        linewidth=2.4,
        label=name,
    )

axes[0].set_title("ODE Residual Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel(r"$\mathcal{L}_{\mathrm{QPINN}}$")
axes[0].legend(loc="best")

axes[1].set_title("Terminal Solution Error")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel(r"$\|x_\theta(T)-x^\star\|_2$")
axes[1].legend(loc="best")

fig.suptitle("Torch QFM-QPINN vs Fully Connected PINN with Lamb", fontsize=13)
plt.show()


# ============================================================
# 11. Final solution comparison
# ============================================================

plt.figure(figsize=(8, 4.8), dpi=140)

component_axis = range(1, n + 1)

plt.plot(
    component_axis,
    x_star.detach().cpu().numpy().flatten(),
    "k--",
    linewidth=2.2,
    label=r"Reference $x^\star$",
)

markers = {
    "QFM-QPINN": "o-",
    "FC-PINN": "s-",
}

for name, info in final_infos.items():
    plt.plot(
        component_axis,
        info["x_T"].detach().cpu().numpy().flatten(),
        markers[name],
        color=colors[name],
        linewidth=2.0,
        markersize=5,
        label=rf"{name}, error={info['x_error']:.2e}",
    )

plt.title("Final Terminal Solution")
plt.xlabel("Component index")
plt.ylabel("Value")
plt.xticks(component_axis)
plt.legend(loc="best")
plt.tight_layout()
plt.show()
